# Improved Math Solver - Single LLM with Enhanced Debugging

This notebook implements the improved math solver that addresses the issues found in the original two-LLM system:

## Key Improvements:
1. **Removed problematic parser LLM** - No more incorrect "short solutions"
2. **Enhanced debugging output** - Full visibility into solver reasoning
3. **Corrected test cases** - Fixed wrong expected answer in Problem 4
4. **Better prompting** - Following WizardMath best practices
5. **Manual verification** - Cross-check all answers with manual calculations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install transformers accelerate torch

In [ ]:
"""
IMPROVED Single-LLM Math Solver with Enhanced Debugging
- Uses only WizardMath (no problematic parser LLM)
- Regex-only parsing for reliability
- Enhanced debugging and logging
- Better prompting following HuggingFace guidelines
- Manual verification of test cases

Key improvements:
1. Removed parser LLM that was generating wrong solutions
2. Added comprehensive debugging output
3. Fixed test case with wrong expected answer (Problem 4: should be 50, not 60)
4. Better prompting strategy for determinant calculations
5. Clear visibility into solver reasoning
"""

import re
import math
import torch
from typing import List, Dict, Any, Optional
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

# Silence HF warnings
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# =========================
# 1) Load WizardMath (single solver) on GPU, fp16
# =========================
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

print("Loading WizardMath tokenizer + model from Google Drive...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_DIR,
        dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    print("WizardMath loaded ✅")
except Exception as e:
    print(f"Error loading WizardMath: {e}")
    raise

# =========================
# 2) Enhanced Solver Pipeline with Better Prompting
# =========================
solver_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,       # Allow more tokens for step-by-step reasoning
    do_sample=False,          # Deterministic for math
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Improved system prompt following WizardMath best practices
SOLVER_SYS = (
    "You are WizardMath, an expert in linear algebra and matrix operations.\n"
    "Solve determinant problems step by step with clear calculations.\n"
    "Always end with: Final Answer: <number>"
)

def solve_with_wizard(question: str, debug: bool = True) -> Dict[str, Any]:
    """
    Solve a math problem with WizardMath and return detailed results.
    """
    # Create a clear, direct prompt
    prompt = f"{SOLVER_SYS}\n\nProblem: {question}\n\nSolution:"
    
    if debug:
        print(f"\n🔍 DEBUG - Prompt sent to solver:")
        print(f"{'='*50}")
        print(prompt)
        print(f"{'='*50}")
    
    try:
        # Generate solution
        output = solver_pipe(prompt, return_full_text=False)[0]["generated_text"]
        raw_output = output.strip()
        
        if debug:
            print(f"\n🔍 DEBUG - Raw solver output:")
            print(f"{'='*50}")
            print(raw_output)
            print(f"{'='*50}")
        
        return {
            "prompt": prompt,
            "raw_output": raw_output,
            "success": True,
            "error": None
        }
    except Exception as e:
        return {
            "prompt": prompt,
            "raw_output": "",
            "success": False,
            "error": str(e)
        }

# =========================
# 3) Regex-Only Parser (No LLM interference)
# =========================
def parse_answer_regex_only(text: str, debug: bool = True) -> Dict[str, Any]:
    """
    Parse the final answer using only regex patterns.
    """
    # Multiple patterns to catch different answer formats
    patterns = [
        r"Final\s*Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"The\s*answer\s*is\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"=\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)(?:\s|$)",
    ]
    
    final_answer = None
    matched_pattern = None
    
    for i, pattern in enumerate(patterns):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            try:
                final_answer = float(match.group(1))
                matched_pattern = f"Pattern {i+1}: {pattern}"
                break
            except (ValueError, TypeError):
                continue
    
    if debug:
        print(f"\n🔍 DEBUG - Parsing results:")
        print(f"Final answer found: {final_answer}")
        print(f"Matched pattern: {matched_pattern}")
    
    return {
        "final_answer": final_answer,
        "matched_pattern": matched_pattern,
        "parsing_success": final_answer is not None
    }

print("🔧 IMPROVED SINGLE-LLM SOLVER LOADED")

In [ ]:
# =========================
# Manual Verification Functions
# =========================
def manual_determinant_2x2(matrix: List[List[float]]) -> float:
    """Calculate 2x2 determinant: ad - bc"""
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

def manual_determinant_3x3(matrix: List[List[float]]) -> float:
    """Calculate 3x3 determinant using cofactor expansion along first row"""
    a, b, c = matrix[0]
    d, e, f = matrix[1] 
    g, h, i = matrix[2]
    
    return (a * (e * i - f * h) - 
            b * (d * i - f * g) + 
            c * (d * h - e * g))

def extract_matrix_from_question(question: str) -> List[List[float]]:
    """Extract matrix values from question text"""
    pattern = r'\[\[(.*?)\]\]'
    match = re.search(pattern, question)
    if not match:
        raise ValueError(f"Could not extract matrix from: {question}")
    
    matrix_str = match.group(1)
    rows = re.split(r'\],\s*\[', matrix_str)
    
    matrix = []
    for row in rows:
        row = row.strip('[]')
        values = [float(x.strip()) for x in row.split(',')]
        matrix.append(values)
    
    return matrix

def get_correct_answer(question: str) -> float:
    """Get the mathematically correct answer for a question"""
    matrix = extract_matrix_from_question(question)
    if len(matrix) == 2:
        return manual_determinant_2x2(matrix)
    elif len(matrix) == 3:
        return manual_determinant_3x3(matrix)
    else:
        raise ValueError(f"Unsupported matrix size: {len(matrix)}x{len(matrix[0])}")

print("✅ Manual verification functions loaded")

In [ ]:
# =========================
# Enhanced Assessment with Debugging
# =========================
def assess_with_debugging(problems: List[Dict[str, Any]], rel_tol: float = 1e-6):
    """
    Assess problems with comprehensive debugging output
    """
    print("🚀 STARTING ENHANCED MATH SOLVER ASSESSMENT")
    print("=" * 60)
    
    results = []
    correct_count = 0
    
    for i, problem in enumerate(problems, 1):
        question = problem["question"]
        given_answer = float(problem["given_answer"])
        
        print(f"\n📝 PROBLEM {i}")
        print(f"Question: {question}")
        print(f"Given expected answer: {given_answer}")
        
        # Get mathematically correct answer
        try:
            correct_answer = get_correct_answer(question)
            print(f"✅ Manual calculation: {correct_answer}")
            
            # Check if given answer is correct
            if not math.isclose(given_answer, correct_answer, rel_tol=rel_tol):
                print(f"⚠️  WARNING: Given expected answer ({given_answer}) is WRONG!")
                print(f"    Correct answer should be: {correct_answer}")
        except Exception as e:
            print(f"❌ Could not verify manually: {e}")
            correct_answer = given_answer  # Fall back to given answer
        
        # Solve with WizardMath
        solver_result = solve_with_wizard(question, debug=True)
        
        if not solver_result["success"]:
            print(f"❌ Solver failed: {solver_result['error']}")
            results.append({
                "problem_num": i,
                "question": question,
                "correct_answer": correct_answer,
                "solver_answer": None,
                "is_correct": False,
                "error": solver_result["error"]
            })
            continue
        
        # Parse the answer
        parse_result = parse_answer_regex_only(solver_result["raw_output"], debug=True)
        
        if not parse_result["parsing_success"]:
            print(f"❌ Could not parse answer from solver output")
            print(f"Raw output was: {solver_result['raw_output']}")
            results.append({
                "problem_num": i,
                "question": question,
                "correct_answer": correct_answer,
                "solver_answer": None,
                "is_correct": False,
                "error": "Parsing failed"
            })
            continue
        
        solver_answer = parse_result["final_answer"]
        is_correct = math.isclose(solver_answer, correct_answer, rel_tol=rel_tol)
        
        print(f"\n📊 RESULTS:")
        print(f"Solver answer: {solver_answer}")
        print(f"Correct answer: {correct_answer}")
        print(f"Status: {'✅ CORRECT' if is_correct else '❌ INCORRECT'}")
        
        if is_correct:
            correct_count += 1
        
        results.append({
            "problem_num": i,
            "question": question,
            "correct_answer": correct_answer,
            "solver_answer": solver_answer,
            "is_correct": is_correct,
            "error": None
        })
    
    # Final summary
    total_problems = len(problems)
    accuracy = (correct_count / total_problems) * 100
    
    print(f"\n🎯 FINAL SUMMARY")
    print(f"=" * 60)
    print(f"Total problems: {total_problems}")
    print(f"Correct answers: {correct_count}")
    print(f"Accuracy: {accuracy:.1f}%")
    
    return results

print("✅ Assessment function loaded")

In [ ]:
# =========================
# Test Problems (with corrected expected answers)
# =========================
sample_problems = [
    {"question": "Compute the determinant of the matrix [[8, 10], [-11, -5]]", "given_answer": 70},
    {"question": "Compute the determinant of the matrix [[-2, 1], [7, -1]]", "given_answer": -5},
    {"question": "Calculate the determinant of the matrix [[4, 6, 2], [6, -6, -5], [3, 3, -1]]", "given_answer": 102},
    {"question": "Calculate the determinant of the matrix [[0, 2, 6], [-4, -4, -5], [3, 0, 1]]", "given_answer": 50},  # CORRECTED: was 60, should be 50
    {"question": "Calculate the determinant of the matrix [[-7, -4, 6], [0, -4, -7], [-7, 1, 4]]", "given_answer": -301},
]

print("📋 Test problems loaded:")
for i, problem in enumerate(sample_problems, 1):
    print(f"{i}. {problem['question']} (Expected: {problem['given_answer']})")

In [ ]:
# =========================
# Run the Enhanced Assessment
# =========================
print("🎯 RUNNING IMPROVED MATH SOLVER ASSESSMENT")
print("\nKey improvements over original:")
print("- ❌ Removed problematic parser LLM")
print("- 🔍 Enhanced debugging output")
print("- ✅ Corrected test case expected answers")
print("- 🎯 Better prompting strategy")
print("- 🔬 Manual verification of all answers")

results = assess_with_debugging(sample_problems)

print("\n" + "="*60)
print("ASSESSMENT COMPLETE!")
print("Check the detailed output above to see:")
print("- Exact prompts sent to the solver")
print("- Raw solver outputs (no parser interference)")
print("- Manual verification of each answer")
print("- Clear identification of any remaining issues")